In [ ]:
from open_vocab_mot import VERI_DATASET_PATH, VERI_DATASET_SIDECAR_PATH
print(VERI_DATASET_PATH, VERI_DATASET_SIDECAR_PATH)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhyudaya12/veri-vehicle-re-identification-dataset", output_dir=str(VERI_DATASET_PATH.parent))

print("Path to dataset files:", path)

In [ ]:
from IPython.display import display
from torchvision.transforms.v2.functional import to_pil_image

In [ ]:
from open_vocab_mot.data.veri_video_ds import VeRiVideoDataset, VeRiSplit


In [ ]:
train_ds = VeRiVideoDataset(
    VERI_DATASET_PATH,
    main_split=VeRiSplit.GALLERY,
    sidecar_root=VERI_DATASET_SIDECAR_PATH,
    load_image_pil=True,
    load_image_tensor=False,
    load_segmentations=True,
    verbose=True
)

In [ ]:
test_sample = train_ds[2]
test_sample

In [ ]:
display(test_sample.frame)
display(to_pil_image(test_sample.segmentation_tensor))

In [ ]:
from open_vocab_mot.data import VideoReIDKPFBatchIterableDataset, VideoReIDItem, collate_video_reid_ds, VideoReIDBatch

In [ ]:
sampler_ds = VideoReIDKPFBatchIterableDataset(
    train_ds,
    batches_per_epoch=100,
    num_identities_per_batch=1,
    num_sequences_per_identity=2,
    num_frames_per_sequence=2,
    allow_resampling_sample_indices=True,
    verbose=True
)

In [ ]:
from torch.utils.data import DataLoader
sampler_loader = DataLoader(sampler_ds, batch_size=None, collate_fn=collate_video_reid_ds)

In [ ]:
test_batch: VideoReIDBatch = next(iter(sampler_loader))

In [ ]:
test_batch

In [ ]:
for i in range(len(test_batch.sample_indices)):
    display(test_batch.frames[i])
    display(to_pil_image(test_batch.segmentations[i]))

In [ ]:
test_ds = VeRiVideoDataset(
    VERI_DATASET_PATH,
    main_split=VeRiSplit.QUERY,
    sidecar_root=VERI_DATASET_SIDECAR_PATH,
    load_image_pil=True,
    load_image_tensor=False,
    load_segmentations=True,
    verbose=True,
    collapse_sequences=True,
)

test_sampler_ds = VideoReIDKPFBatchIterableDataset(
    test_ds,
    batches_per_epoch=100,
    num_identities_per_batch=1,
    num_sequences_per_identity=1,
    num_frames_per_sequence=4,
    allow_resampling_sample_indices=True,
    verbose=True
)

test_loader = DataLoader(test_sampler_ds, batch_size=None, collate_fn=collate_video_reid_ds)

In [ ]:
test_batch: VideoReIDBatch = next(iter(test_loader))
test_batch

In [ ]:
for i in range(len(test_batch.sample_indices)):
    display(test_batch.frames[i])
    display(to_pil_image(test_batch.segmentations[i]))